In [9]:
from transformers import BertForQuestionAnswering
from transformers import BertTokenizer
import torch

In [13]:
model_name = 'bert-large-uncased-whole-word-masking-finetuned-squad'

In [14]:
model = BertForQuestionAnswering.from_pretrained(model_name)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-large-uncased-whole-word-masking-finetuned-squad
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
tokenizer = BertTokenizer.from_pretrained(model_name)

In [10]:
sunset_document = "Sunset Motors is a renowned automobile dealership that has been a cornerstone of the automotive industry since its establishment in 1978. Located in the pi cturesque town of Crestwaod, nestled in the heart of California's scenic Central Valley, Sunset Motors has built a reputation for oxcellence, reliability, and customer satisfaction over the past four derades. Founded by visionary entrepreneur Robert Anderson, Sunset Motors began as a humble, family-owned busi ness with a small lot of used cars. However, under Andersan's leadership and commitmont to quality, it quickly ovolved into a thriving dealership offoring a wide range of vehicles from various manufacturers. Today, the dealership spans over 18 acres, showcasing a vast inventory of new and pre-owned cars, truc ks, Ss, and luxury vehicles. One of Sunset Motors' standout features is its dedication to sustaimability. In 2010, the dealership made a landmark decisio n to incorporate environmentally friendly practices, including solar panels to power the facility, energy-efficient lighting, and a comprehensive recycling program. This commitment to eco-consciousness has earned Sunset Motors recognition as an industry leader in sustainable automotive retsil. Sunset Motors pr oudly offers a diverse range of vchicles, including popular brands like Ford, Toyota, Honda, Chevrolet, and BMN, catering to a wide spectrum of tastes and preferences. In addition to its outstanding vehicle selection, Sunset Motors offers flexihle financing options, allowing customers to secure affordable loa ns and leases with competitive interest rates."
sunset_document

"Sunset Motors is a renowned automobile dealership that has been a cornerstone of the automotive industry since its establishment in 1978. Located in the pi cturesque town of Crestwaod, nestled in the heart of California's scenic Central Valley, Sunset Motors has built a reputation for oxcellence, reliability, and customer satisfaction over the past four derades. Founded by visionary entrepreneur Robert Anderson, Sunset Motors began as a humble, family-owned busi ness with a small lot of used cars. However, under Andersan's leadership and commitmont to quality, it quickly ovolved into a thriving dealership offoring a wide range of vehicles from various manufacturers. Today, the dealership spans over 18 acres, showcasing a vast inventory of new and pre-owned cars, truc ks, Ss, and luxury vehicles. One of Sunset Motors' standout features is its dedication to sustaimability. In 2010, the dealership made a landmark decisio n to incorporate environmentally friendly practices, including sola

In [16]:
def faq_bot(question):
    context = sunset_document 
    input_ids = tokenizer.encode(question, context) 
    tokens = tokenizer.convert_ids_to_tokens(input_ids) 
    sep_idx = input_ids.index(tokenizer.sep_token_id) 
    num_seg_a = sep_idx+1 
    num_seg_b = len(input_ids) - num_seg_a 
    segment_ids = [0]*num_seg_a + [1]*num_seg_b 
    output = model(torch.tensor([input_ids]), token_type_ids = torch.tensor([segment_ids])) 
    answer_start = torch.argmax(output.start_logits) 
    answer_end = torch.argmax(output.end_logits) 
    if answer_end >= answer_start: 
        answer = ' '.join(tokens[answer_start: answer_end+1]) 
    else: 
        print("I don't know how to answer this question, can you ask another one?") 
        
    corrected_answer = '' 
    for word in answer.split():
        if word[0:2] == '##':
            corrected_answer += word[2:] 
        else:  
            corrected_answer += ' ' + word
    return corrected_answer 

In [17]:
faq_bot('where is the dealership located')

' crestwaod'

In [18]:
faq_bot('what make of car available')

' new and pre - owned cars , truc ks , ss , and luxury vehicles'

In [19]:
faq_bot('how large is the dealership is')

' 18 acres'